# The Carbon Market MRV Problem and Our Solution

Global warming is accelerating due to excess carbon emissions. To achieve **Net Zero**, countries and corporations are mandated to purchase carbon credits generated by projects that sequester carbon, such as planting mangroves (Blue Carbon). However, verifying exactly how much carbon these projects sequester is currently a massive bottleneck. It requires expensive manual teams on the ground to measure and report.

**Our solution automates MRV (Measurement, Reporting, and Verification)**. By leveraging satellite imagery and spatio-temporal deep learning, we eliminate the need for manual teams. While this notebook focuses on Mangroves (Blue Carbon), this architecture is built to scale across all forms of carbon (Green, White, etc.) globally.

# Synchronizing Spatial and Predictor Data with the Firestore Registry

In [ ]:
!pip install firebase-admin

In [ ]:
import os

target_directory = '/content/drive/MyDrive/STAGE 3'

# Get a list of all files and directories within the target_directory
all_items = os.listdir(target_directory)

# Filter for .csv files that start with 'Patch'
filtered_csv_files = []
for item in all_items:
    full_path = os.path.join(target_directory, item)
    if os.path.isfile(full_path) and item.endswith('.csv') and item.startswith('Patch'):
        filtered_csv_files.append(full_path)

print("Found CSV files starting with 'Patch':")
for f in filtered_csv_files:
    print(f)

Found CSV files starting with 'Patch':
/content/drive/MyDrive/STAGE 3/Patch_0_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_1_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_3_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_4_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_5_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_6_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_7_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_8_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_9_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_10_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_11_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_13_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_12_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_14_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_15_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_16_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_17_Satelite.csv
/content/drive/MyDrive/STAGE 3/Patch_18_Satelite.csv
/content/drive/M

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd
import os
import glob

# Init Firestore connection using service account

if not firebase_admin._apps:
    cred = credentials.Certificate('/content/drive/MyDrive/STAGE 3/mangroove-startup-96309-firebase-adminsdk-fbsvc-44d45acec2_projectmail.json')
    firebase_admin.initialize_app(cred)

db = firestore.client()

In [ ]:
def upload_patches_to_firestore(directory_path):

    patch_files = glob.glob(os.path.join(directory_path, "Patch_*_Satelite.csv"))

    for file_path in patch_files:
        df = pd.read_csv(file_path)
        patch_id = str(df['patch_id'].iloc[0])
        print(f"Processing {patch_id}...")

        # Create a reference to the main patch document
        # Structure: Patches (Col) -> Patch_X (Doc)
        patch_ref = db.collection('Patches').document(f"Patch_{patch_id}")

        # Group data by date (YYYY-MM-DD) to handle the 61-month timeline
        grouped_by_date = df.groupby('date')

        for date_val, date_df in grouped_by_date:
            # We use a sub-collection for dates to avoid hitting the 1MB document limit
            # Structure: Patches -> Patch_X -> TimeSeries (SubCol) -> YYYY-MM (Doc)
            date_doc_id = str(date_val)[:7] # Standardize to YYYY-MM
            date_ref = patch_ref.collection('TimeSeries').document(date_doc_id)

            # Prepare the coordinate map
            pixel_data = {}
            for _, row in date_df.iterrows():
                # Sanitize coordinate string for use as a Firestore key (remove dots/slashes)
                coord_key = str(row['coordinate']).replace('.', '_').replace('/', '-')

                # Extract all features into a dictionary
                features = row.drop(['coordinate', 'date', 'patch_id']).to_dict()
                pixel_data[coord_key] = features

            # Use a batch to upload to ensure atomicity and speed
            batch = db.batch()
            batch.set(date_ref, {"mangrove_pixels": pixel_data}, merge=True)
            batch.commit()

    print("All patches successfully synced to Firestore.")

# Execute the upload
upload_patches_to_firestore("/content/drive/MyDrive/STAGE 3/")

Processing 0...
Processing 1...
Processing 3...
Processing 4...
Processing 5...
Processing 6...
Processing 7...
Processing 8...
Processing 9...
Processing 10...
Processing 11...
Processing 13...
Processing 12...
Processing 14...
Processing 15...
Processing 16...
Processing 17...
Processing 18...
Processing 19...
Processing 21...
Processing 20...
Processing 22...
Processing 23...
Processing 24...
Processing 25...
Processing 26...
Processing 27...
Processing 28...
Processing 29...
Processing 30...
Processing 31...
Processing 32...
Processing 33...
Processing 34...
Processing 36...
Processing 37...
Processing 39...
Processing 40...
Processing 41...
Processing 43...
Processing 42...
Processing 45...
Processing 44...
Processing 46...
Processing 47...
Processing 48...
Processing 50...
Processing 52...
Processing 53...
Processing 54...
Processing 55...
Processing 56...
Processing 57...
Processing 58...
Processing 59...
Processing 60...
Processing 61...
Processing 62...
Processing 63...
Proces

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/STAGE 1/graph_edges.csv')
df.columns

Index(['source', 'target', 'edge_type', 'distance_km', 'source_latitude',
       'source_longitude', 'target_latitude', 'target_longitude'],
      dtype='object')

In [ ]:
import pandas as pd
from firebase_admin import firestore

db = firestore.client()

def upload_graph_edges(csv_path):
    edges_df = pd.read_csv(csv_path)
    print(f"Syncing {len(edges_df)} graph edges to Firestore...")

    # Group by source to minimize document access
    grouped_edges = edges_df.groupby('source_patch_id')

    for source_id, group in grouped_edges:
        # Reference the source patch
        source_ref = db.collection('Patches').document(f"Patch_{int(source_id)}")

        # Batch write for performance
        batch = db.batch()

        for _, row in group.iterrows():
            dest_id = int(row['destination_patch_id'])
            # Create a unique ID for the edge in the sub-collection
            edge_ref = source_ref.collection('Connections').document(f"To_Patch_{dest_id}")

            edge_data = {
                "destination_id": dest_id,
                "edge_type": row['edge_type'],
                "distance_km": float(row['distance_km']),
                "metadata": {
                    "source_id": int(source_id),
                    "is_directed": True if row['edge_type'] == 'tidal' else False
                }
            }
            batch.set(edge_ref, edge_data)

        batch.commit()
        print(f"Edges for Patch {source_id} successfully uploaded.")

# Execute the upload
upload_graph_edges('/content/drive/MyDrive/STAGE 3/graph_edges.csv')

Syncing 79 graph edges to Firestore...
Edges for Patch 4 successfully uploaded.
Edges for Patch 8 successfully uploaded.
Edges for Patch 9 successfully uploaded.
Edges for Patch 10 successfully uploaded.
Edges for Patch 13 successfully uploaded.
Edges for Patch 14 successfully uploaded.
Edges for Patch 18 successfully uploaded.
Edges for Patch 19 successfully uploaded.
Edges for Patch 20 successfully uploaded.
Edges for Patch 24 successfully uploaded.
Edges for Patch 25 successfully uploaded.
Edges for Patch 26 successfully uploaded.
Edges for Patch 29 successfully uploaded.
Edges for Patch 30 successfully uploaded.
Edges for Patch 31 successfully uploaded.
Edges for Patch 32 successfully uploaded.
Edges for Patch 33 successfully uploaded.
Edges for Patch 34 successfully uploaded.
Edges for Patch 35 successfully uploaded.
Edges for Patch 36 successfully uploaded.
Edges for Patch 37 successfully uploaded.
Edges for Patch 39 successfully uploaded.
Edges for Patch 40 successfully uploaded

# Deriving Spatial Boundaries and Centroids for Registry Ingestion

In [ ]:
df1=pd.read_csv('/content/drive/MyDrive/STAGE 1/clustered_patches_with_polygons_dbscan.csv')
df1.head()
print(df1.patch_id.unique())
print(df1.columns)
#

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73]
Index(['patch_id', 'monthly_absorption_tCO2e_ha', 'polygon_coords'], dtype='object')


In [ ]:
df3=df1[['patch_id','polygon_coords']]
df3.head()

,patch_id,polygon_coords
0,0,"[[23.95947323405253, 52.0624604136], [23.95947..."
1,1,"[[23.982271489212007, 51.9181264344], [23.9803..."
2,2,"[[24.115261310975608, 52.832241636], [24.12286..."
3,3,"[[24.12856029315197, 53.909516988], [24.130460..."
4,4,"[[24.17605665806754, 54.1458900264], [24.16085..."


In [ ]:
df3.shape

(74, 2)

In [ ]:
from shapely.geometry import Polygon

print("Shapely Polygon imported.")

Shapely Polygon imported.


In [ ]:
import ast

def calculate_centroid(coord_string):
    try:
        # Safely evaluate the string to a list of lists
        coords = ast.literal_eval(coord_string)
        # Create a Polygon object from the coordinates
        polygon = Polygon(coords)
        # Calculate the centroid and return its x, y coordinates
        return (polygon.centroid.x, polygon.centroid.y)
    except Exception as e:
        print(f"Error processing coordinates: {coord_string} - {e}")
        return None

df3['centroid_coordinates'] = df3['polygon_coords'].apply(calculate_centroid)

print(df3.head())

Error processing coordinates: [] - UnsupportedOperationException: getX called on empty Point

   patch_id                                     polygon_coords  \
0         0  [[23.95947323405253, 52.0624604136], [23.95947...   
1         1  [[23.982271489212007, 51.9181264344], [23.9803...   
2         2  [[24.115261310975608, 52.832241636], [24.12286...   
3         3  [[24.12856029315197, 53.909516988], [24.130460...   
4         4  [[24.17605665806754, 54.1458900264], [24.16085...   

                       centroid_coordinates  
0   (23.962428563425057, 52.05153658586666)  
1    (23.98488723104793, 51.91403378848696)  
2       (24.119694305034397, 52.8336361672)  
3  (24.127836539019924, 53.916912983828574)  
4  (24.173321702549323, 54.129477466892304)  


In [ ]:
import ast

def calculate_centroid(coord_string):
    try:
        # Safely evaluate the string to a list of lists
        coords = ast.literal_eval(coord_string)

        # Check if coordinates list is empty
        if not coords:
            return None # Return None for empty polygons

        # Create a Polygon object from the coordinates
        polygon = Polygon(coords)
        # Calculate the centroid and return its x, y coordinates
        return (polygon.centroid.x, polygon.centroid.y)
    except Exception as e:
        print(f"Error processing coordinates: {coord_string} - {e}")
        return None

df3['centroid_coordinates'] = df3['polygon_coords'].apply(calculate_centroid)

print(df3.head())

   patch_id                                     polygon_coords  \
0         0  [[23.95947323405253, 52.0624604136], [23.95947...   
1         1  [[23.982271489212007, 51.9181264344], [23.9803...   
2         2  [[24.115261310975608, 52.832241636], [24.12286...   
3         3  [[24.12856029315197, 53.909516988], [24.130460...   
4         4  [[24.17605665806754, 54.1458900264], [24.16085...   

                       centroid_coordinates  
0   (23.962428563425057, 52.05153658586666)  
1    (23.98488723104793, 51.91403378848696)  
2       (24.119694305034397, 52.8336361672)  
3  (24.127836539019924, 53.916912983828574)  
4  (24.173321702549323, 54.129477466892304)  


In [ ]:
patch_geometry_data = []
for index, row in df3.iterrows():
    patch_data = {
        'patch_id': int(row['patch_id']),
        'polygon_coordinates': row['polygon_coords'],
        'centroid_coordinates': row['centroid_coordinates']
    }
    patch_geometry_data.append(patch_data)

print(f"Prepared {len(patch_geometry_data)} patch geometry records for Firestore.")
# print first 5 elements to verify
for i in range(min(5, len(patch_geometry_data))):
    print(patch_geometry_data[i])

Prepared 74 patch geometry records for Firestore.
{'patch_id': 0, 'polygon_coordinates': '[[23.95947323405253, 52.0624604136], [23.95947323405253, 52.0561850232], [23.963272943245777, 52.045726039200005], [23.967072652439025, 52.0415424456], [23.95947323405253, 52.0624604136]]', 'centroid_coordinates': (23.962428563425057, 52.05153658586666)}
{'patch_id': 1, 'polygon_coordinates': '[[23.982271489212007, 51.9181264344], [23.980371634615384, 51.9139428408], [23.982271489212007, 51.911851044], [23.98607119840525, 51.9097592472], [23.989870907598497, 51.9139428408], [23.987971053001875, 51.9160346376], [23.982271489212007, 51.9181264344]]', 'centroid_coordinates': (23.98488723104793, 51.91403378848696)}
{'patch_id': 2, 'polygon_coordinates': '[[24.115261310975608, 52.832241636], [24.122860729362102, 52.832241636], [24.12096087476548, 52.8364252296], [24.115261310975608, 52.832241636]]', 'centroid_coordinates': (24.119694305034397, 52.8336361672)}
{'patch_id': 3, 'polygon_coordinates': '[[2

In [ ]:
def upload_patch_geometry_to_firestore(geometry_data_list):
    print(f"Syncing {len(geometry_data_list)} patch geometry records to Firestore...")

    for patch_data in geometry_data_list:
        patch_id = patch_data['patch_id']
        polygon_coords = patch_data['polygon_coordinates']
        centroid_coords = patch_data['centroid_coordinates']

        # Reference the existing patch document
        patch_ref = db.collection('Patches').document(f"Patch_{patch_id}")

        # Prepare data to update (add polygon and centroid)
        update_data = {
            "polygon_coordinates": polygon_coords,
            "centroid_coordinates": centroid_coords
        }

        # Use set with merge=True to add these fields without overwriting the entire document
        patch_ref.set(update_data, merge=True)
        print(f"Updated geometry for Patch_{patch_id}.")

    print("All patch geometry records successfully synced to Firestore.")

# Execute the upload
upload_patch_geometry_to_firestore(patch_geometry_data)

Syncing 74 patch geometry records to Firestore...
Updated geometry for Patch_0.
Updated geometry for Patch_1.
Updated geometry for Patch_2.
Updated geometry for Patch_3.
Updated geometry for Patch_4.
Updated geometry for Patch_5.
Updated geometry for Patch_6.
Updated geometry for Patch_7.
Updated geometry for Patch_8.
Updated geometry for Patch_9.
Updated geometry for Patch_10.
Updated geometry for Patch_11.
Updated geometry for Patch_12.
Updated geometry for Patch_13.
Updated geometry for Patch_14.
Updated geometry for Patch_15.
Updated geometry for Patch_16.
Updated geometry for Patch_17.
Updated geometry for Patch_18.
Updated geometry for Patch_19.
Updated geometry for Patch_20.
Updated geometry for Patch_21.
Updated geometry for Patch_22.
Updated geometry for Patch_23.
Updated geometry for Patch_24.
Updated geometry for Patch_25.
Updated geometry for Patch_26.
Updated geometry for Patch_27.
Updated geometry for Patch_28.
Updated geometry for Patch_29.
Updated geometry for Patch_30.

# PATCH Wise Monthly Carbon Absorbsion


In [ ]:
import pandas as pd
import glob
import os

def calculate_total_monthly_absorption(directory_path):
    total_monthly_absorption_data = []

    # Find all relevant CSV files
    patch_files = glob.glob(os.path.join(directory_path, "Patch_*_Satelite.csv"))

    for file_path in patch_files:
        df = pd.read_csv(file_path)

        # Extract patch_id (assuming it's consistent within each file)
        if not df.empty:
            patch_id = df['patch_id'].iloc[0]
        else:
            print(f"Warning: {file_path} is empty, skipping.")
            continue

        # Group by date and sum 'monthly_absorption_tCO2e_ha'
        monthly_summary = df.groupby('date')['monthly_absorption_tCO2e_ha'].sum().reset_index()

        for index, row in monthly_summary.iterrows():
            total_monthly_absorption_data.append({
                'patch_id': patch_id,
                'date': row['date'],
                'total_absorption_tCO2e_ha': row['monthly_absorption_tCO2e_ha']
            })

    return total_monthly_absorption_data

# Execute the function with the appropriate directory
# Assuming '/content/drive/MyDrive/STAGE 3/' is the correct path from previous steps
directory_for_patches = '/content/drive/MyDrive/STAGE 3/'
absorption_results = calculate_total_monthly_absorption(directory_for_patches)

print(f"Calculated total monthly absorption for {len(absorption_results)} entries.")
# Print first 10 results to verify
for i in range(min(10, len(absorption_results))):
    print(absorption_results[i])


Calculated total monthly absorption for 4148 entries.
{'patch_id': np.int64(0), 'date': '2021-01-01', 'total_absorption_tCO2e_ha': 6.186348892607056}
{'patch_id': np.int64(0), 'date': '2021-02-01', 'total_absorption_tCO2e_ha': 6.196812770506557}
{'patch_id': np.int64(0), 'date': '2021-03-01', 'total_absorption_tCO2e_ha': 4.450016970436002}
{'patch_id': np.int64(0), 'date': '2021-04-01', 'total_absorption_tCO2e_ha': 4.816182455379178}
{'patch_id': np.int64(0), 'date': '2021-05-01', 'total_absorption_tCO2e_ha': 4.857774274647961}
{'patch_id': np.int64(0), 'date': '2021-06-01', 'total_absorption_tCO2e_ha': 4.816365389041876}
{'patch_id': np.int64(0), 'date': '2021-07-01', 'total_absorption_tCO2e_ha': 8.527143185471378}
{'patch_id': np.int64(0), 'date': '2021-08-01', 'total_absorption_tCO2e_ha': 4.904265687627033}
{'patch_id': np.int64(0), 'date': '2021-09-01', 'total_absorption_tCO2e_ha': 7.420109658805738}
{'patch_id': np.int64(0), 'date': '2021-10-01', 'total_absorption_tCO2e_ha': 7.393

In [ ]:
from firebase_admin import firestore
import firebase_admin # Ensure firebase_admin is imported if not already in this scope

# db object should already be initialized from previous steps
# if not firebase_admin._apps:
#     # Re-initialize Firestore if it somehow got lost (unlikely in Colab session)
#     # cred = credentials.Certificate('/content/drive/MyDrive/STAGE 3/mangroove-startup-96309-firebase-adminsdk-fbsvc-44d45acec2_projectmail.json')
#     # firebase_admin.initialize_app(cred)
# db = firestore.client()

def upload_monthly_absorption_to_firestore(absorption_data):
    print(f"Syncing {len(absorption_data)} monthly absorption records to Firestore...")

    # Group data by patch_id to use batch writes more effectively
    grouped_by_patch = {}
    for entry in absorption_data:
        patch_id = entry['patch_id']
        if patch_id not in grouped_by_patch:
            grouped_by_patch[patch_id] = []
        grouped_by_patch[patch_id].append(entry)

    for patch_id, monthly_records in grouped_by_patch.items():
        batch = db.batch()
        patch_ref = db.collection('Patches').document(f"Patch_{int(patch_id)}")

        for record in monthly_records:
            date_val = record['date']
            # Standardize to YYYY-MM for the document ID within TimeSeries subcollection
            date_doc_id = str(date_val)[:7]
            time_series_doc_ref = patch_ref.collection('TimeSeries').document(date_doc_id)

            # Prepare update data for the specific monthly absorption field
            update_data = {
                'total_absorption_tCO2e_ha': record['total_absorption_tCO2e_ha']
            }
            # Use set with merge=True to add or update this field without overwriting other data
            batch.set(time_series_doc_ref, update_data, merge=True)

        batch.commit()
        print(f"Uploaded monthly absorption for Patch_{int(patch_id)}.")

    print("All monthly absorption records successfully synced to Firestore.")

# Execute the upload function with the previously generated absorption_results
upload_monthly_absorption_to_firestore(absorption_results)


Syncing 4148 monthly absorption records to Firestore...
Uploaded monthly absorption for Patch_0.
Uploaded monthly absorption for Patch_1.
Uploaded monthly absorption for Patch_3.
Uploaded monthly absorption for Patch_4.
Uploaded monthly absorption for Patch_5.
Uploaded monthly absorption for Patch_6.
Uploaded monthly absorption for Patch_7.
Uploaded monthly absorption for Patch_8.
Uploaded monthly absorption for Patch_9.
Uploaded monthly absorption for Patch_10.
Uploaded monthly absorption for Patch_11.
Uploaded monthly absorption for Patch_13.
Uploaded monthly absorption for Patch_12.
Uploaded monthly absorption for Patch_14.
Uploaded monthly absorption for Patch_15.
Uploaded monthly absorption for Patch_16.
Uploaded monthly absorption for Patch_17.
Uploaded monthly absorption for Patch_18.
Uploaded monthly absorption for Patch_19.
Uploaded monthly absorption for Patch_21.
Uploaded monthly absorption for Patch_20.
Uploaded monthly absorption for Patch_22.
Uploaded monthly absorption f

In [ ]:
import firebase_admin
from firebase_admin import firestore

# Ensure db is initialized, if not already
if not firebase_admin._apps:
    # This part should ideally be handled by credentials.Certificate if running standalone
    # For Colab, assuming it's initialized from previous steps
    print("Firebase Admin not initialized, please ensure credentials are set up.")
    # Example placeholder for initialization if needed:
    # cred = firebase_admin.credentials.Certificate('/content/drive/MyDrive/STAGE 3/mangroove-startup-96309-firebase-adminsdk-fbsvc-44d45acec2_projectmail.json')
    # firebase_admin.initialize_app(cred)

# Get the Firestore client (assuming 'db' is already globally available from previous steps)
db = firestore.client()

# 1. Choose a representative patch_id and date
chosen_patch_id = 0
chosen_date_month = '2021-01' # YYYY-MM

# 2. Construct the Firestore document reference
patch_doc_ref = db.collection('Patches').document(f"Patch_{chosen_patch_id}")
time_series_doc_ref = patch_doc_ref.collection('TimeSeries').document(chosen_date_month)

# 3. Retrieve the document
doc = time_series_doc_ref.get()

# 4. Check if the retrieved document exists and contains the field
if doc.exists:
    data = doc.to_dict()
    print(f"Document for Patch_{chosen_patch_id} in {chosen_date_month} exists.")
    if 'total_absorption_tCO2e_ha' in data:
        firestore_absorption_value = data['total_absorption_tCO2e_ha']
        print(f"  'total_absorption_tCO2e_ha': {firestore_absorption_value}")
        print("  Other data in this document:")
        for key, value in data.items():
            if key != 'total_absorption_tCO2e_ha':
                # Print only first 50 characters of nested dicts to keep output clean
                if isinstance(value, dict):
                    print(f"    '{key}': {{... (contains {len(value)} entries)}}")
                else:
                    print(f"    '{key}': {value}")

        # 5. Optionally, compare with absorption_results variable
        # Find the corresponding entry in absorption_results
        expected_value = None
        for entry in absorption_results:
            if entry['patch_id'] == chosen_patch_id and entry['date'] == f"{chosen_date_month}-01":
                expected_value = entry['total_absorption_tCO2e_ha']
                break

        if expected_value is not None:
            print(f"\n  Expected value from `absorption_results`: {expected_value}")
            if abs(firestore_absorption_value - expected_value) < 1e-9: # Comparing floats
                print("  Value in Firestore matches the calculated value (within tolerance).")
            else:
                print("  WARNING: Value in Firestore does NOT exactly match the calculated value.")
        else:
            print(f"  Could not find corresponding entry for Patch_{chosen_patch_id} {chosen_date_month} in `absorption_results`.")

    else:
        print("  'total_absorption_tCO2e_ha' field not found in the document.")
else:
    print(f"Document for Patch_{chosen_patch_id} in {chosen_date_month} does not exist.")

Document for Patch_0 in 2021-01 exists.
  'total_absorption_tCO2e_ha': 6.186348892607056
  Other data in this document:
    'mangrove_pixels': {... (contains 5 entries)}

  Expected value from `absorption_results`: 6.186348892607056
  Value in Firestore matches the calculated value (within tolerance).


# PATCH Health Score



`health_score = total_absorption_tCO2e_ha + average_NDVI + average_GEDI_canopy_height_rh100`


In [ ]:
from firebase_admin import firestore
import pandas as pd

db = firestore.client()

def calculate_and_upload_health_score():
    print("Starting health score calculation and upload...")

    patches_ref = db.collection('Patches')
    patches = patches_ref.stream()

    for patch in patches:
        patch_id_str = patch.id.replace('Patch_', '')
        patch_id = int(patch_id_str)

        time_series_ref = patch.reference.collection('TimeSeries')
        time_series_docs = time_series_ref.stream()

        batch = db.batch()
        updates_count = 0

        for ts_doc in time_series_docs:
            data = ts_doc.to_dict()
            date_month = ts_doc.id # YYYY-MM

            total_absorption = data.get('total_absorption_tCO2e_ha')
            mangrove_pixels = data.get('mangrove_pixels', {}) # This should be a dictionary of pixel data

            if total_absorption is None or not mangrove_pixels:
                # Skip if essential data is missing for this month
                continue

            # Extract NDVI and GEDI_canopy_height_rh100 from all pixels
            ndvis = []
            gedi_heights = []
            for pixel_data in mangrove_pixels.values():
                if 'NDVI' in pixel_data:
                    ndvis.append(pixel_data['NDVI'])
                if 'GEDI_canopy_height_rh100' in pixel_data:
                    gedi_heights.append(pixel_data['GEDI_canopy_height_rh100'])

            avg_ndvi = sum(ndvis) / len(ndvis) if ndvis else 0
            avg_gedi_height = sum(gedi_heights) / len(gedi_heights) if gedi_heights else 0

            # Calculate health score
            health_score = total_absorption + avg_ndvi + avg_gedi_height

            # Prepare update for the TimeSeries document
            update_data = {
                'health_score': health_score,
                'average_NDVI': avg_ndvi,
                'average_GEDI_canopy_height_rh100': avg_gedi_height
            }
            batch.update(ts_doc.reference, update_data)
            updates_count += 1

            # Commit batch every 500 operations or at the end of a patch's time series
            if updates_count % 499 == 0 and updates_count > 0:
                batch.commit()
                batch = db.batch()

        # Commit any remaining operations in the batch for this patch
        if updates_count > 0:
            batch.commit()
        print(f"Processed Patch_{patch_id}. Updated {updates_count} monthly health scores.")

    print("Health score calculation and upload completed.")

# Execute the function
calculate_and_upload_health_score()

Starting health score calculation and upload...
Processed Patch_0. Updated 61 monthly health scores.
Processed Patch_1. Updated 61 monthly health scores.
Processed Patch_10. Updated 61 monthly health scores.
Processed Patch_11. Updated 61 monthly health scores.
Processed Patch_12. Updated 61 monthly health scores.
Processed Patch_13. Updated 61 monthly health scores.
Processed Patch_14. Updated 61 monthly health scores.
Processed Patch_15. Updated 61 monthly health scores.
Processed Patch_16. Updated 61 monthly health scores.
Processed Patch_17. Updated 61 monthly health scores.
Processed Patch_18. Updated 61 monthly health scores.
Processed Patch_19. Updated 61 monthly health scores.
Processed Patch_2. Updated 0 monthly health scores.
Processed Patch_20. Updated 61 monthly health scores.
Processed Patch_21. Updated 61 monthly health scores.
Processed Patch_22. Updated 61 monthly health scores.
Processed Patch_23. Updated 61 monthly health scores.
Processed Patch_24. Updated 61 monthly

In [ ]:
from firebase_admin import firestore

db = firestore.client()

# 1. Choose a representative patch_id and date to verify
chosen_patch_id_verify = 0
chosen_date_month_verify = '2021-01' # YYYY-MM

# 2. Construct the Firestore document reference
patch_doc_ref_verify = db.collection('Patches').document(f"Patch_{chosen_patch_id_verify}")
time_series_doc_ref_verify = patch_doc_ref_verify.collection('TimeSeries').document(chosen_date_month_verify)

# 3. Retrieve the document
doc_verify = time_series_doc_ref_verify.get()

# 4. Check if the retrieved document exists and contains the new fields
if doc_verify.exists:
    data_verify = doc_verify.to_dict()
    print(f"Document for Patch_{chosen_patch_id_verify} in {chosen_date_month_verify} exists.")

    # Check for health_score
    if 'health_score' in data_verify:
        firestore_health_score = data_verify['health_score']
        print(f"  'health_score': {firestore_health_score}")
    else:
        print("  'health_score' field not found in the document.")

    # Check for average_NDVI
    if 'average_NDVI' in data_verify:
        firestore_avg_ndvi = data_verify['average_NDVI']
        print(f"  'average_NDVI': {firestore_avg_ndvi}")
    else:
        print("  'average_NDVI' field not found in the document.")

    # Check for average_GEDI_canopy_height_rh100
    if 'average_GEDI_canopy_height_rh100' in data_verify:
        firestore_avg_gedi = data_verify['average_GEDI_canopy_height_rh100']
        print(f"  'average_GEDI_canopy_height_rh100': {firestore_avg_gedi}")
    else:
        print("  'average_GEDI_canopy_height_rh100' field not found in the document.")

    print("  Other data in this document (truncated for brevity):")
    for key, value in data_verify.items():
        if key not in ['health_score', 'average_NDVI', 'average_GEDI_canopy_height_rh100']:
            if isinstance(value, dict):
                print(f"    '{key}': {{... (contains {len(value)} entries)}}")
            else:
                print(f"    '{key}': {value}")

    # Optional: Re-calculate and compare with expected values if `absorption_results` and `data` (from previous cell) are available
    # For this verification, we'll manually check against the logic applied in the previous step.
    # To do a full comparison, we would need to re-fetch the raw pixel data and calculate averages and the sum.
    # As the previous cell confirmed individual absorption values, and the logic was just run,
    # presence of the new fields with non-zero values implies success.

else:
    print(f"Document for Patch_{chosen_patch_id_verify} in {chosen_date_month_verify} does not exist.")

Document for Patch_0 in 2021-01 exists.
  'health_score': 9.325748562671505
  'average_NDVI': 0.08383116573095316
  'average_GEDI_canopy_height_rh100': 3.055568504333496
  Other data in this document (truncated for brevity):
    'total_absorption_tCO2e_ha': 6.186348892607056
    'mangrove_pixels': {... (contains 5 entries)}


# REGISTER Collection

In [ ]:
registry_raw_data = []

# Get a reference to the 'Patches' collection
patches_ref = db.collection('Patches')

# Define the date range for filtering
start_date_str = '2021-01'
end_date_str = '2025-12'

# Iterate through each document in the 'Patches' collection
for patch_doc in patches_ref.stream():
    # Extract patch_id from the document ID
    patch_id = patch_doc.id.replace('Patch_', '')

    # Get a reference to the 'TimeSeries' subcollection for the current patch
    time_series_ref = patch_doc.reference.collection('TimeSeries')

    # Iterate through all documents in the 'TimeSeries' subcollection for the current patch
    for ts_doc in time_series_ref.stream():
        data = ts_doc.to_dict()
        date_month = ts_doc.id # Document ID is the date in 'YYYY-MM' format

        # Filter documents based on the date range in Python
        if start_date_str <= date_month <= end_date_str:
            # Extract total_absorption_tCO2e_ha if it exists
            total_absorption = data.get('total_absorption_tCO2e_ha')

            if total_absorption is not None:
                registry_raw_data.append({
                    'patch_id': patch_id,
                    'date': date_month,
                    'total_absorption_tCO2e_ha': total_absorption
                })

print(f"Extracted {len(registry_raw_data)} monthly absorption records for registry creation.")
# Print first few elements to verify
for i in range(min(5, len(registry_raw_data))):
    print(registry_raw_data[i])

Extracted 4080 monthly absorption records for registry creation.
{'patch_id': '0', 'date': '2021-01', 'total_absorption_tCO2e_ha': 6.186348892607056}
{'patch_id': '0', 'date': '2021-02', 'total_absorption_tCO2e_ha': 6.196812770506557}
{'patch_id': '0', 'date': '2021-03', 'total_absorption_tCO2e_ha': 4.450016970436002}
{'patch_id': '0', 'date': '2021-04', 'total_absorption_tCO2e_ha': 4.816182455379178}
{'patch_id': '0', 'date': '2021-05', 'total_absorption_tCO2e_ha': 4.857774274647961}


In [ ]:
registry_df = pd.DataFrame(registry_raw_data)

print(f"Created DataFrame with {len(registry_df)} rows and {len(registry_df.columns)} columns.")
print(registry_df.head())

Created DataFrame with 4080 rows and 3 columns.
  patch_id     date  total_absorption_tCO2e_ha
0        0  2021-01                   6.186349
1        0  2021-02                   6.196813
2        0  2021-03                   4.450017
3        0  2021-04                   4.816182
4        0  2021-05                   4.857774


# Registry Collection Creation in FireStore

In [ ]:
import datetime

registry_prepared_data = []

# Get the current UTC timestamp
current_timestamp = datetime.datetime.utcnow()

# Iterate through each row of the registry_df DataFrame
for index, row in registry_df.iterrows():
    patch_id_str = str(row['patch_id'])
    date_id_str = str(row['date'])

    record = {
        'carbonAbsorption': row['total_absorption_tCO2e_ha'],
        'dateId': date_id_str,
        'id': f"Patch_{patch_id_str}_{date_id_str}",
        'patchId': f"Patch_{patch_id_str}",
        'registryDate': current_timestamp,
        'status': 'Active',
        'verraStatus': 'Pending'
    }
    registry_prepared_data.append(record)

print(f"Prepared {len(registry_prepared_data)} registry records.")
# Print the first 5 elements to verify
for i in range(min(5, len(registry_prepared_data))):
    print(registry_prepared_data[i])

/tmp/ipython-input-1882281875.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_timestamp = datetime.datetime.utcnow()


Prepared 4080 registry records.
{'carbonAbsorption': 6.186348892607056, 'dateId': '2021-01', 'id': 'Patch_0_2021-01', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 49, 350614), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 6.196812770506557, 'dateId': '2021-02', 'id': 'Patch_0_2021-02', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 49, 350614), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 4.450016970436002, 'dateId': '2021-03', 'id': 'Patch_0_2021-03', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 49, 350614), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 4.816182455379178, 'dateId': '2021-04', 'id': 'Patch_0_2021-04', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 49, 350614), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 4.857774274647961, 'dateId': '2021-05', 'id': 'Patch_0_2021

**Reasoning**:
The previous code generated a DeprecationWarning for `datetime.datetime.utcnow()`. I will update the code to use the recommended `datetime.datetime.now(datetime.UTC)` to fix this warning and ensure future compatibility.



In [ ]:
import datetime

registry_prepared_data = []

# Get the current UTC timestamp using the recommended method
current_timestamp = datetime.datetime.now(datetime.UTC)

# Iterate through each row of the registry_df DataFrame
for index, row in registry_df.iterrows():
    patch_id_str = str(row['patch_id'])
    date_id_str = str(row['date'])

    record = {
        'carbonAbsorption': row['total_absorption_tCO2e_ha'],
        'dateId': date_id_str,
        'id': f"Patch_{patch_id_str}_{date_id_str}",
        'patchId': f"Patch_{patch_id_str}",
        'registryDate': current_timestamp,
        'status': 'Active',
        'verraStatus': 'Pending'
    }
    registry_prepared_data.append(record)

print(f"Prepared {len(registry_prepared_data)} registry records.")
# Print the first 5 elements to verify
for i in range(min(5, len(registry_prepared_data))):
    print(registry_prepared_data[i])

Prepared 4080 registry records.
{'carbonAbsorption': 6.186348892607056, 'dateId': '2021-01', 'id': 'Patch_0_2021-01', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 57, 844720, tzinfo=datetime.timezone.utc), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 6.196812770506557, 'dateId': '2021-02', 'id': 'Patch_0_2021-02', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 57, 844720, tzinfo=datetime.timezone.utc), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 4.450016970436002, 'dateId': '2021-03', 'id': 'Patch_0_2021-03', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 57, 844720, tzinfo=datetime.timezone.utc), 'status': 'Active', 'verraStatus': 'Pending'}
{'carbonAbsorption': 4.816182455379178, 'dateId': '2021-04', 'id': 'Patch_0_2021-04', 'patchId': 'Patch_0', 'registryDate': datetime.datetime(2026, 2, 20, 12, 17, 57, 844720, tzinfo=datetime.timezone.utc), 'statu

**Reasoning**:
The `registry_prepared_data` has been successfully structured, and the next step is to upload this data to a new 'Registry' collection in Firestore. I will define a function to iterate through the prepared data and use Firestore batch writes for efficient uploading.



In [ ]:
from firebase_admin import firestore

db = firestore.client()

def upload_registry_to_firestore(registry_data_list):
    print(f"Syncing {len(registry_data_list)} registry records to Firestore...")

    batch = db.batch()
    operations_count = 0
    batch_size = 500 # Firestore recommends batches of up to 500 operations

    for record in registry_data_list:
        # Use the 'id' field as the document ID in the 'Registry' collection
        doc_id = record['id']
        doc_ref = db.collection('Registry').document(doc_id)

        # Firestore does not directly support datetime objects with timezone info in the client library.
        # Convert to a naive datetime object or string if not already handled by the library.
        # The Python client library typically handles datetime objects with timezone automatically,
        # but it's good practice to be aware if issues arise.
        # For simplicity, we'll pass it as is, assuming the client handles it.

        batch.set(doc_ref, record)
        operations_count += 1

        # Commit batch periodically
        if operations_count % batch_size == 0:
            batch.commit()
            batch = db.batch()
            print(f"Committed {operations_count} records...")

    # Commit any remaining operations in the final batch
    if operations_count % batch_size != 0 or operations_count == 0: # Ensure final commit even if count is 0
        batch.commit()

    print(f"All {operations_count} registry records successfully synced to Firestore.")

# Execute the upload function with the prepared data
upload_registry_to_firestore(registry_prepared_data)

Syncing 4080 registry records to Firestore...
Committed 500 records...
Committed 1000 records...
Committed 1500 records...
Committed 2000 records...
Committed 2500 records...
Committed 3000 records...
Committed 3500 records...
Committed 4000 records...
All 4080 registry records successfully synced to Firestore.
